In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from MD_utils import read_lammpstrj, plot_equilibration

# MD Data Analysis

## Exercise 0: Set Parameter, Equilibrate & Load Data

### Simulation parameters

In [ ]:
rho_lj =  ???    #!!!!!!!!!!!!!
dt_lj = 0.005
dt_sampling = 10
Nbox = 6

### Conversion to Real Units Parameters (Argon)

In [ ]:
sigma_a = 3.4
sigma_m = sigma_a * 1e-10
eps_joule = 120 * 1.380649e-23
mass_kg = 39.948 * 1.66054e-27
tau_s = sigma_m * (mass_kg / eps_joule) ** 0.5  # in seconds
tau_ps = tau_s * 1e12         # convert to picoseconds
dt_ps = dt_lj * tau_ps        # in picoseconds

Lbox_a = ((4/rho_lj)**(1./3.))* sigma_a * Nbox # Lbox in angstroms

### Remove equilibration

Choose when the system "look" in equilibrium

In [ ]:
# Define your equilibration and run sets
Nframes_eq =  ???                            #!!!!!!!!!!!!!
plot_equilibration(file_name="thermo_???.dat", Nframes_eq = Nframes_eq, rho_lj=rho_lj )         #!!!!!!!!!!!!!

## Exercise 1: MSD

### Load Unwrapped Trajectory

In [ ]:
utraj_lj = read_lammpstrj(file_name = "unwrap_pos_???.lammpstrj")    #!!!!!!!!!!!!!
utraj_lj.shape

### Define Parameters

In [ ]:
Nframes_prod = utraj_lj.shape[0] - Nframes_eq

Natoms = utraj_lj.shape[1]

### Remove Equilibration & Convert to Real Units

In [ ]:
utraj_lj = utraj_lj[Nframes_eq:]

utraj_a = utraj_lj*sigma_a

Complete the function below, in order to compute the mean-square displacement.

In [ ]:
def compute_msd(unwrapped):
    '''
    Compute the mean squared displacement!
    '''
    r0 = unwrapped[0]               # initial positions

    displacement = unwrapped - r0  # displacements from initial position

    #square_displacement =                               

    mean_square_displacement = ???   # mean squared-displacements

    return mean_square_displacement


Compute the MSD and plot it!

In [ ]:
msd = compute_msd(utraj_a)

In [ ]:
time_ps = np.arange(Nframes_prod) * dt_ps  * dt_sampling # time array in ps
plt.plot(time_ps, msd , label='MSD (Argon)')
plt.xlabel("Time [ps]")
plt.ylabel("MSD [$\\mathrm{Å}^2$]")
plt.title("Mean Square Displacement of Argon")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## To discuss with your nearest neighbors:
- What shape of the MSD do you expect for different phases?
- With your current data, could you reduce the noise of the plot? Justify your answer


# Exercise 2: VACF

### Load Velocities

In [ ]:
vels_lj = read_lammpstrj("vels_???.lammpstrj")  #!!!!!!!!!!!!!

### Remove Equilibration & Convert to Real Units

In [ ]:
vels_lj = vels_lj[Nframes_eq:]
vels_apps = vels_lj * sigma_a / tau_ps

Complete the function below, in order to compute the velocity autocorrelation-function.

In [ ]:
def compute_vacf(vels):
    '''
    Compute the velocity autocorrelation function
    '''
    max_tau = int(Nframes_prod/2)
    vacf = np.zeros(max_tau)

    for tau in range(max_tau):         # Loop over tau
        dot_sum = 0.0                  # sum of all dot products for a given tau

        for t in range(Nframes_prod - tau):  # Loop over initial frame
            v0 = vels[t]
            vtau = vels[t + tau]
            dot_sum += ???                      

        vacf[tau] = dot_sum / normalization  # The normalization should be the total number of dot products that you are summing
    return vacf

Compute VACF and plot it!

In [ ]:
vacf=compute_vacf(vels_apps)

In [ ]:
maxN_plot = 25                                      #!!!!!!!!!!!!!
time_ps = np.arange(Nframes_prod) * dt_ps  * dt_sampling # time array in ps

plt.plot(time_ps[:maxN_plot], vacf[:maxN_plot])       
ax = plt.gca()
ax.xaxis.set_major_locator(ticker.MaxNLocator(10))  # up to 10 x-ticks
ax.yaxis.set_major_locator(ticker.MaxNLocator(10))  # up to 10 y-ticks
plt.grid(True)
plt.xlabel("τ [ps]")
plt.ylabel("VACF [$\\mathrm{Å}^2/ps^2$]")
plt.grid(True)
plt.title("Velocity Autocorrelation Function")
plt.tight_layout()
plt.show()

## To discuss with your nearest neighbors:
- What shape would you expect for the VACF of a system in the solid, liquid, or gaseous state? 
- What is the meaning of negative values of the VACF?
- What would be a system whose vacf(τ) = 1?

# Exercise 3: g(r)

### Load Wrapped Trajectory

In [ ]:
wtraj_lj = read_lammpstrj("wrap_pos_???.lammpstrj")    #!!!!!!!!!!!!!

### Remove Equilibration & Convert to Real Units

In [ ]:
wtraj_lj = wtraj_lj[Nframes_eq:]
wtraj_a = wtraj_lj*sigma_a

### Parameters

In [ ]:
Nbins = 512
rmax_a = Lbox_a/2
Lbin_a = rmax_a/Nbins
rs_a = Lbin_a*np.arange(Nbins)  # Array for the plot

### 3.1 Count Pair Distances (done)

In [ ]:
def count_pair_distances_firstNframes(n_frames, trajectory):
    '''
    Accumulate pairwise distances into distance_counter (minimum image convention)
    for the first n_frames frames of the given trajectory.
    '''
    pair_distance_counts=np.zeros(Nbins)
    for i in range(n_frames):
        rx = trajectory[i][:, 0]
        ry = trajectory[i][:, 1]
        rz = trajectory[i][:, 2]

        for k in range(Natoms - 1):
            j = k + 1

            # Distances to atoms with superior index (not normalized)
            dx = rx[k] - rx[j:Natoms]
            dy = ry[k] - ry[j:Natoms]
            dz = rz[k] - rz[j:Natoms]

            # Apply minimum image convention
            dx = dx / Lbox_a
            dy = dy / Lbox_a
            dz = dz / Lbox_a

            dx = dx - np.rint(dx)
            dy = dy - np.rint(dy)
            dz = dz - np.rint(dz)

            dx = dx * Lbox_a
            dy = dy * Lbox_a
            dz = dz * Lbox_a

            # dx, dy, dz now with minimum image convention in real units
            r2 = dx * dx + dy * dy + dz * dz
            r = np.sqrt(r2)

            for corrected_distance in r:
                if corrected_distance <= rmax_a:
                    pair_distance_counts[int(corrected_distance / Lbin_a)] += 2
    return pair_distance_counts

In [ ]:
pair_distance_counts = count_pair_distances_firstNframes(Nframes_prod, wtraj_a)

In [ ]:
plt.plot(rs_a,pair_distance_counts, label="histogram")

# Reference r^2 curve, scaled to match the tail of the histogram
scale = pair_distance_counts[-1] / rs_a[-1]**2
plt.plot(rs_a, scale * rs_a**2, '--', label=r"$\propto r^2$")
plt.xlabel("r [Å]")
plt.ylabel("g(r)")
plt.grid(True)
plt.legend()
plt.title("Pair Distance histogram (Unnormalized g(r))")
plt.tight_layout()
plt.show()

### 3.2 Find Normalization 

Find the normalization such that g(r) = pair_distance_counts/normalization

In [ ]:
normalization = np.ones(Nbins)         
                                    #!!!!!!!!!!!!!

In [ ]:
plt.plot(rs_a,pair_distance_counts/normalization)
plt.xlabel("r [Å]")
plt.ylabel("g(r)")
plt.grid(True)
plt.title("g(r)")
plt.tight_layout()
plt.show()

## To discuss with your nearest neighbors:
- Why the pair_distance_counts seems grows as r^2?
- With you current data, how would you count the typical number of atoms in the first "shell"? What about the typical number of atoms in the second "shell"? 

### Appendix

In [ ]:
D_a2pps = ???
print(f"Your D = {D_a2pps*10:.3f} × 10⁻⁵ cm² s⁻¹ (Rahman: 2.43 × 10⁻⁵ cm² s⁻¹ at (rho_lj, ⟨T⟩_prod)=(0.814, 0.787))")

In [ ]:
random_array = np.ones(Nbins)
for i in range(Nbins):
    random_array[i] = (4/3) * np.pi* ((Lbin_a * (i+1))**3 - (Lbin_a * i)**3)